In [8]:
import torch
import warnings

# Suppress non-critical warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

device =  "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.manual_seed(42)
print(f"Using device: {device}")

Using device: cuda


In [9]:
from datasets import load_dataset
import os

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
os.makedirs("data", exist_ok=True)
dataset.to_json("data/dolly_train.json", orient="records", lines=True)

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

12982756

In [10]:
import pandas as pd
sampledata=pd.DataFrame(dataset)
sampledata

,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,,Tope,classification
2,Why can camels survive for long without water?,,Camels use the fat in their humps to keep them...,open_qa
3,"Alice's parents have three daughters: Amy, Jes...",,The name of the third daughter is Alice,open_qa
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981.",closed_qa
...,...,...,...,...
15006,How do i accept the change,,Embrace the change and see the difference,brainstorming
15007,What is a laser and who created it?,A laser is a device that emits light through a...,A laser is a device that emits light from an e...,summarization
15008,What is the difference between a road bike and...,,Road bikes are built to be ridden on asphalt a...,open_qa
15009,How does GIS help in the real estate investmen...,,"Real estate investors depend on precise, accur...",general_qa


In [11]:
# 1. Load Pretrained model & tokenizer
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [12]:
# Converting raw text into training samples
def create_text_column(example):

    text = f"""### Instruction:{example['instruction']}
               ### Context:{example.get('context','')}
               ### Response: {example['response']}"""

    return {"text": text}

dataset = load_dataset("json", data_files="data/dolly_train.json")["train"]
dataset = dataset.shuffle(seed=42).select(range(1000))  # reduce steps + memory
dataset = dataset.map(
    create_text_column,
    # Do not remove original columns here; SFTTrainer will handle further processing.
    # The 'text' column will be added.
)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
# Injecting LoRA into the Transformer (Core Step)
from peft import LoraConfig, get_peft_model

# LoRA config
lora_config = LoraConfig(
    r=8,# rank controls — How much the model can deviate from its original behavior

    lora_alpha=16,
    # Scaling the update - LoRA adds a small extra update to the existing weight
    # lora_alpha decides how big that update is.
    # alpha too small- update barely changes base model and viceversa

    target_modules=["q_proj", "v_proj"],
    #decide which parts of the Transformer are allowed to learn during fine-tuning

    lora_dropout=0.05, # Regularizing the adoption

    bias="none", # bias="none" keeps all bias terms frozen, only Lora matrices are trained

    task_type="CAUSAL_LM", #  saying focus on next word prediction task
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### Prep the model for training

In [14]:

model.enable_input_require_grads()# input embeddings don’t track gradients (they are  along with weights)
#This line ensures that even though the input embeddings aren't directly trained, gradients can still flow through them to the trainable LoRA parameters.

model.gradient_checkpointing_enable() #Saves memory by not storing everything; it recalculates some values when needed.
#Memoization trades memory for time (store results to save computation time).
#Gradient checkpointing trades time for memory (re-compute activations to save memory).

model.config.use_cache = False # Turns off caching so gradients are calculated correctly while training.

### Defining the Supervised Fine-Tuning Setup

In [15]:
from transformers import TrainingArguments

# Training arguments
training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=False, # Changed to False for CPU compatibility
    logging_steps=5,
)

###Running LoRA + SFT Training

In [16]:
from transformers import TrainingArguments

# Training arguments
training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=5,
    dataloader_pin_memory=False,
)

In [18]:
!pip install trl
from trl import SFTTrainer

model.to(device)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
    # tokenizer=tokenizer, # Removed as it's an unexpected argument
    # dataset_text_field='text' # Removed as it's an unexpected argument
    # max_seq_length=512, # Removed as it's an unexpected argument without explicit tokenizer
)
trainer.train()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 21.4 MB/s eta 0:00:00


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (4115 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
5,2.262496
10,2.138377
15,2.057280
20,1.940046
25,1.814300
30,1.765054
35,1.601777
40,1.729770
45,1.728634
50,1.789697


TrainOutput(global_step=125, training_loss=1.7786126861572265, metrics={'train_runtime': 123.7208, 'train_samples_per_second': 8.083, 'train_steps_per_second': 1.01, 'total_flos': 1949515444887552.0, 'train_loss': 1.7786126861572265})

In [19]:
# Saving the LoRA Adapter
trainer.model.save_pretrained("lora-adapter")
tokenizer.save_pretrained("lora-adapter")
# Ensure adapter_config.json and adapter_model.bin are in 'lora-adapter' directory

('lora-adapter/tokenizer_config.json',
 'lora-adapter/chat_template.jinja',
 'lora-adapter/tokenizer.json')

### Merging LoRA with the Base Model+ inferenceing

In [20]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base_model, "lora-adapter", local_files_only=True)

# Merge LoRA weights into the base model
merged_model = model.merge_and_unload()

# Save the merged model
merged_model.save_pretrained("merged_model_new")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("merged_model_new", local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

inputs = tokenizer(
    "Explain LoRA fine-tuning in simple terms for a product manager.",
    return_tensors="pt"
).to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    do_sample=False
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output)

Explain LoRA fine-tuning in simple terms for a product manager.
Explain LoRA fine-tuning in simple terms for a product manager.
LoRA (Low-Rate Wireless) is a wireless communication technology that uses LoRaWAN (LoRa Wide Area Network) technology. LoRaWAN is a standardized protocol for low-power wide-area wireless communication. LoRaWAN is a low-power, long-range wireless communication technology that is designed to be used in IoT applications. LoRaWAN is a low-power, long-range wireless communication technology that is designed to be used in IoT applications.
LoRaWAN is a low-power, long-range wireless communication technology that is designed to be used in IoT applications. LoRaWAN is a low-power, long-range wireless communication technology that is designed to be used in IoT applications. LoRaWAN is a low-power, long-range wireless communication technology that is designed to be used in IoT applications. LoRaWAN is a low-power, long-range wireless communication technology that is des